# Lesson 16 · Thermal energy transfer

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/StativeSh/quantum-playground/blob/main/lessons/16_thermal_energy_transfer.ipynb)

**For:** A-level Physics (for example AQA 3.6.2.1, OCR A 5.1.3, Edexcel Topic 9) and IB Physics (B.1 thermal energy transfers). Newton's law of cooling in section 5 is extension material. Check your own specification for the exact wording.

**Before this lesson:** Lesson 9, *Ideal gases and kinetic theory* (internal energy and the kinetic model of matter). Also assumed: energy, power and E = Pt.

**Leads to:** the end of this route for now.

**You will be able to:**

1. describe internal energy as the sum of the random kinetic and potential energies of the molecules
2. use $Q = mc\Delta\theta$ for a temperature change and $Q = mL$ for a change of state
3. read a heating curve: explain its flat sections and why its sloping sections have different gradients
4. find the final temperature of a mixture from energy lost = energy gained, including when ice melts
5. use Newton's law of cooling, $T = T_a + (T_i - T_a)e^{-kt}$, and explain why cooling slows down (extension)

**How to use this lesson:** run each cell with Shift+Enter. Where you see *Try it*, change a number and run the cell again. Before some experiments a **Predict** box asks you to commit to an answer; write it down, run the cell, then answer the **Explain** box below it. Answers to both are at the very end, after the worked solutions. Exercises are at the end, with worked solutions below them.

In [ ]:
# Install the course API in a fresh Colab runtime or upgrade an older installation.
import importlib.util
import subprocess
import sys
from importlib.metadata import version

if importlib.util.find_spec("qplay") is None or int(version("quantum-playground").split(".")[0]) < 1:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "quantum-playground>=1,<2"], check=True)

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from qplay import thermal, numerics

c_ice, c_water, L_f = 2100, 4200, 334e3    # J/(kg K), J/(kg K), J/kg

## 1. Internal energy and temperature

The molecules of any substance move about at random and pull on one another. The **internal energy** of an object is the sum of the random **kinetic** energies and the **potential** energies of all its molecules.

- **Temperature** is a measure of the average random kinetic energy of the molecules. Heat an object without changing its state and the molecules move faster, so its temperature rises.
- During a **change of state** (melting, boiling) the energy supplied separates the molecules against their attractions. Their potential energy rises but their average kinetic energy does not, so the temperature stays constant even though energy keeps going in.

Energy flows from hotter to colder objects. A temperature *difference* of 1 K is the same as 1 °C, so in $\Delta\theta$ you can use either.

## 2. Specific heat capacity

The **specific heat capacity** $c$ of a substance is the energy needed to raise the temperature of 1 kg of it by 1 K. Heating a mass $m$ through $\Delta\theta$ needs

$$Q = mc\Delta\theta .$$

Water's specific heat capacity, about $4200\ \mathrm{J\,kg^{-1}\,K^{-1}}$, is unusually large; metals have much smaller values. The next cell gives the same 10 kJ to 1 kg of three materials.

In [ ]:
energy, mass = 10e3, 1.0                   # J, kg
for name, c in [("water", 4200), ("aluminium", 900), ("copper", 390)]:
    print(f"{name:>9}: c = {c:4d} J/(kg K), temperature rise = Q/(mc) = {energy / (mass * c):5.1f} K")

The same energy warms copper more than ten times as much as water. That is why a metal pan heats up quickly and why the sea changes temperature so slowly through the year.

## 3. The heating curve and latent heat

A change of state needs energy without any change of temperature. The **specific latent heat** $L$ is the energy needed to change the state of 1 kg without changing its temperature:

$$Q = mL .$$

For melting ice, $L_f = 334\ \mathrm{kJ\,kg^{-1}}$, the specific latent heat of **fusion**. `thermal.heating_curve` works out the temperature after a given amount of energy has gone into a sample that starts as a solid below its melting point. It uses kelvin, so ice melts at 273.15 K.

**Predict (P1).** A 1.0 kW heater supplies energy steadily to 1.0 kg of ice that starts at 263 K ($-10$ °C). Sketch a graph of temperature against time for the first 12 minutes. Is there any stage where the temperature stays constant even though the heater is on? Write your prediction down before you run the next cell.

In [ ]:
power, mass, start, melt = 1000.0, 1.0, 263.15, 273.15   # W, kg, K, K
t = np.linspace(0, 720, 721)                             # seconds
T = thermal.heating_curve(power * t, mass, start, melt, c_ice, L_f, c_water)

plt.figure(figsize=(8, 3.8))
plt.plot(t / 60, T - 273.15)
plt.axhline(0, color="0.7", lw=0.6)
plt.xlabel("time / minutes"); plt.ylabel("temperature / °C")
plt.title("1.0 kg of ice heated at 1.0 kW")

warm_ice = mass * c_ice * (melt - start)                 # energy to reach the melting point
melting = mass * L_f                                     # energy to melt it all
print(f"warming the ice to 0 °C: {warm_ice / 1e3:.0f} kJ, taking {warm_ice / power:.0f} s")
print(f"melting it at 0 °C:      {melting / 1e3:.0f} kJ, taking {melting / power:.0f} s")

**Explain (P1).** Energy goes in at a steady 1.0 kW throughout. Where is it going while the temperature is flat? Use the ideas of section 1.

**Predict (P2).** Compare the two sloping parts of the graph. For each joule supplied, which warms faster, the ice before it melts or the water afterwards? By what factor?

In [ ]:
q = np.linspace(0, 420e3, 4201)                          # energy supplied / J
T = thermal.heating_curve(q, mass, start, melt, c_ice, L_f, c_water)
slope = np.gradient(T, q)                                # kelvin per joule

ice, water = q < 15e3, q > 360e3
print(f"ice:   {slope[ice].mean() * 1e3:.3f} K per kJ  (1/(m c_ice)   = {1e3 / (mass * c_ice):.3f})")
print(f"water: {slope[water].mean() * 1e3:.3f} K per kJ  (1/(m c_water) = {1e3 / (mass * c_water):.3f})")
print(f"ratio: {slope[ice].mean() / slope[water].mean():.2f}")

plt.figure(figsize=(8, 3.8))
plt.plot(q / 1e3, T - 273.15)
plt.xlabel("energy supplied / kJ"); plt.ylabel("temperature / °C")
plt.title("The same heating curve against energy: gradient = 1/(mc)");

**Explain (P2).** On a graph of temperature against energy the gradient is $\Delta\theta/Q$. Use $Q = mc\Delta\theta$ to explain the ratio.

*Try it:* change `mass` to 2.0 kg and run both cells again. What happens to the length of the flat section?

## 4. Mixtures: energy lost = energy gained

When hot and cold things are mixed in an insulated container, energy flows until they reach the same final temperature $T$. If no energy escapes,

$$\text{energy lost by the hot parts} = \text{energy gained by the cold parts}.$$

For two lots of water this gives a straight equation for $T$. When ice is involved it is easier to write one function, the **net energy gained** by everything in the mixture for a trial final temperature, and find the temperature that makes it zero. `numerics.bisect` does that by repeatedly halving an interval that contains the answer.

In [ ]:
def two_waters(T):
    # 0.20 kg of water at 80 °C poured into 0.30 kg at 20 °C
    return 0.20 * c_water * (T - 80) + 0.30 * c_water * (T - 20)

def ice_in_drink(T):
    # 20 g of ice at 0 °C dropped into 0.25 kg of water at 25 °C (all the ice melts)
    gained_by_ice = 0.020 * L_f + 0.020 * c_water * (T - 0)
    gained_by_drink = 0.25 * c_water * (T - 25)          # negative: the drink loses energy
    return gained_by_ice + gained_by_drink

print(f"two waters: T = {numerics.bisect(two_waters, 0, 100):.2f} °C")
print(f"ice in a drink: T = {numerics.bisect(ice_in_drink, 0, 25):.2f} °C")
print(f"20 g of water at 0 °C instead of ice: T = {numerics.bisect(lambda T: 0.020 * c_water * T + 0.25 * c_water * (T - 25), 0, 25):.2f} °C")

Most of the cooling comes from **melting** the ice, not from warming the meltwater: 20 g of water at 0 °C would cool the drink only to about 23 °C, but 20 g of ice takes it to about 17 °C.

## 5. Newton's law of cooling (extension)

A hot object in cooler surroundings loses energy at a rate roughly proportional to how much hotter it is than its surroundings. The excess temperature then falls exponentially:

$$T = T_a + (T_i - T_a)\,e^{-kt},$$

where $T_a$ is the ambient (room) temperature, $T_i$ the initial temperature and $k$ a constant for the object and its surroundings. `thermal.cooling` evaluates this.

**Predict (P3).** A drink at 80 °C cools in a 20 °C room. Does it lose the same number of degrees in each 10-minute interval? If not, does it lose more at the start or later?

In [ ]:
T_room, T_start, k = 20.0, 80.0, 0.0010    # °C, °C, per second
t = np.arange(0, 3601, 600)                # every 10 minutes for an hour
T = thermal.cooling(t, T_start, T_room, k)
for (t0, a), (t1, b) in zip(zip(t, T), zip(t[1:], T[1:])):
    print(f"{t0 / 60:3.0f}-{t1 / 60:3.0f} min: {a:5.1f} -> {b:5.1f} °C, lost {a - b:4.1f} K, excess kept {(b - T_room) / (a - T_room):.3f}")

fine = np.linspace(0, 3600, 361)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.6))
ax1.plot(fine / 60, thermal.cooling(fine, T_start, T_room, k))
ax1.axhline(T_room, color="k", ls=":", label="room temperature")
ax1.set_xlabel("time / minutes"); ax1.set_ylabel("temperature / °C"); ax1.legend(fontsize=8)
ax2.semilogy(fine / 60, thermal.cooling(fine, T_start, T_room, k) - T_room)
ax2.set_yticks([2, 5, 10, 20, 50], ["2", "5", "10", "20", "50"]); ax2.minorticks_off()
ax2.set_xlabel("time / minutes"); ax2.set_ylabel("excess over room / K (log scale)")
fig.suptitle("A drink cooling from 80 °C in a 20 °C room");

**Explain (P3).** Look at the last column. What stays the same from one interval to the next? Why is the right-hand graph a straight line?

**Common mistakes**

- Using the mass of only one part of a mixture, or forgetting that melted ice must then be **warmed** as water.
- Forgetting the latent heat when a substance changes state.
- Using the specific heat capacity of water for ice: ice's is about half.
- Mixing joules and kilojoules: $L_f = 334\ \mathrm{kJ\,kg^{-1}} = 334\,000\ \mathrm{J\,kg^{-1}}$.

---

## Exercises

Take $c_{\text{water}} = 4200\ \mathrm{J\,kg^{-1}\,K^{-1}}$, $c_{\text{ice}} = 2100\ \mathrm{J\,kg^{-1}\,K^{-1}}$ and $L_f = 334\ \mathrm{kJ\,kg^{-1}}$. Assume no energy is lost to the surroundings except in Q6. Give answers to 3 significant figures.

**Q1.** Find the energy needed to heat 2.0 kg of water from 15 °C to 100 °C.

**Q2.** A 2.2 kW kettle supplies the energy in Q1. How long does it take?

**Q3.** Find the energy needed to melt 0.50 kg of ice that is already at 0 °C.

**Q4.** Find the total energy needed to turn 0.50 kg of ice at −10 °C into water at 20 °C.

**Q5.** A 20 g ice cube at 0 °C is dropped into 0.25 kg of water at 25 °C in an insulated cup. Find the final temperature in °C.

**Q6 (simulation).** Coffee at 80 °C cools in a 20 °C room with $k = 0.0010\ \mathrm{s^{-1}}$. Use `numerics.bisect` with `thermal.cooling` to find how long it takes to reach 50 °C, then check your answer with a formula.

In [ ]:
def check(name, yours, correct, tol=0.01):
    if yours is None:
        print(f"{name}: not answered yet")
    elif abs(yours - correct) <= tol * abs(correct):
        print(f"{name}: correct")
    else:
        print(f"{name}: not yet, check your working (you gave {yours})")

# The correct values are computed here from the physics, not typed in.
_answers = {
    "Q1": 2.0 * c_water * (100 - 15),
    "Q2": 2.0 * c_water * (100 - 15) / 2200,
    "Q3": 0.50 * L_f,
    "Q4": 0.50 * c_ice * 10 + 0.50 * L_f + 0.50 * c_water * 20,
    "Q5": numerics.bisect(lambda T: 0.020 * L_f + 0.020 * c_water * T + 0.25 * c_water * (T - 25), 0, 25),
    "Q6": numerics.bisect(lambda t: thermal.cooling(t, 80, 20, 0.0010) - 50, 0, 1e4),
}

# Replace None with your answers, in SI units unless the question says otherwise.
mine = {"Q1": None, "Q2": None, "Q3": None, "Q4": None, "Q5": None, "Q6": None}
for q in mine:
    check(q, mine[q], _answers[q])

---

## Worked solutions

Try the questions before reading on.

**Q1.** $Q = mc\Delta\theta = 2.0 \times 4200 \times 85 = 7.14\times10^{5}$ J.

**Q2.** $t = Q/P = 7.14\times10^{5}/2200 = 325$ s, about 5.4 minutes.

**Q3.** $Q = mL_f = 0.50 \times 334\,000 = 1.67\times10^{5}$ J.

**Q4.** Three stages: warm the ice, $0.50 \times 2100 \times 10 = 10\,500$ J; melt it, $167\,000$ J; warm the water, $0.50 \times 4200 \times 20 = 42\,000$ J. Total $2.195\times10^{5}$ J. Melting takes three-quarters of it. The heating curve agrees, below.

**Q5.** Energy gained by the ice: $0.020 \times 334\,000 + 0.020 \times 4200 \times T$. Energy lost by the water: $0.25 \times 4200 \times (25 - T)$. Setting them equal: $6680 + 84T = 26\,250 - 1050T$, so $T = 19\,570/1134 = 17.3$ °C.

**Q6.** $50 = 20 + 60e^{-kt}$, so $e^{-kt} = \tfrac12$ and $t = \ln 2/k = 693$ s, about 12 minutes. The excess over room temperature halves every 693 s, however hot the coffee starts.

In [ ]:
for q, value in _answers.items():
    print(f"{q}: {value:#.3g}".rstrip("."))

print(f"check Q4: the heating curve after {_answers['Q4']:.0f} J reads "
      f"{thermal.heating_curve(_answers['Q4'], 0.50, 263.15, 273.15, c_ice, L_f, c_water):.2f} K (20 °C = 293.15 K)")
print(f"check Q6: ln 2 / k = {np.log(2) / 0.0010:.1f} s")

### Predict and explain: answers

**P1.** Yes. For the first 21 s the ice warms from $-10$ °C to 0 °C; then the temperature stays at 0 °C for 334 s (over five minutes) while the ice melts; then the water warms. During the flat section the energy increases the **potential** energy of the molecules, breaking them free of the rigid solid structure, so their average kinetic energy, and hence the temperature, does not change.

**P2.** The ice warms twice as fast per joule. The gradient is $\Delta\theta/Q = 1/mc$, and the specific heat capacity of ice (2100) is half that of water (4200).

**P3.** No. It loses most at the start (about 27 K in the first 10 minutes, only about 1.3 K in the last 10). What stays the same is the **fraction** of the excess over room temperature kept in each interval, $e^{-600k} \approx 0.549$. A quantity that falls by the same factor in equal times is exponential, and the logarithm of an exponential is a straight line.